In [ ]:
# -*- coding: utf-8 -*-
"""
CÓDIGO ÚNICO — RF PONTO A PONTO vs PARK
VARREDURA DE FAIXAS + HISTOGRAMAS + HEATMAPS 2D + HEATMAPS 3D COM TEMPERATURA

Como usar:
1) Coloque este arquivo na mesma pasta do arquivo:
       base-completo--.pkl

2) Ajuste, se quiser:
       FREQ_MIN_GLOBAL_KHZ
       FREQ_MAX_GLOBAL_KHZ
       LARGURA_FAIXA_KHZ
       PASSO_FAIXA_KHZ

3) Rode tudo.

Para testar de 10 em 10 kHz:
       LARGURA_FAIXA_KHZ = 10
       PASSO_FAIXA_KHZ = 10

Para testar de 20 em 20 kHz:
       LARGURA_FAIXA_KHZ = 20
       PASSO_FAIXA_KHZ = 20

Para testar de 30 em 30 kHz:
       LARGURA_FAIXA_KHZ = 30
       PASSO_FAIXA_KHZ = 30
"""


# -*- coding: utf-8 -*-
"""
VARREDURA DE FAIXAS DE FREQUÊNCIA — RF PONTO A PONTO vs PARK

Objetivo:
1) Rodar apenas dois métodos:
   - Random Forest direto ponto a ponto
   - Park

2) Para várias faixas de frequência, calcular:
   - RMSD
   - CCDM

3) Gerar histogramas grandes, com fontes maiores, comparando RF e Park.

4) Descobrir automaticamente qual faixa parece melhor para análise de dano,
   considerando:
   - erro baixo no estado sem dano
   - separação entre dano 0, dano 1 e dano 2
   - penalização se a ordem esperada D0 < D1 < D2 for invertida

Autor: adaptado para Luiz Eduardo Abdala José
"""

# ============================================================
# 1) IMPORTS
# ============================================================
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 2) PARÂMETROS GERAIS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

# Temperatura de referência usada para montar a curva referência saudável
REF_TEMP = 30

# Pasta de saída
PASTA_SAIDA = "resultados_varredura_rf_park"
os.makedirs(PASTA_SAIDA, exist_ok=True)

# Suavização das curvas compensadas
SMOOTH_WIN = 5

# Park
PARK_MAX_SHIFT_FRAC = 0.25
PARK_SMOOTH_WIN = 5
PARK_NSTEPS = 151   # aumente para 201 se quiser mais precisão, mas fica mais lento

# Random Forest ponto a ponto
RF_COMP_POINT_PARAMS = dict(
    n_estimators=250,       # se ficar lento, use 150; se quiser mais robusto, use 400
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0,
)

# Varredura de faixas
# Exemplo: se usar largura 10 e passo 5, testa 30-40, 35-45, 40-50, etc.
FREQ_MIN_GLOBAL_KHZ = 10
FREQ_MAX_GLOBAL_KHZ = 100
LARGURA_FAIXA_KHZ = 20
PASSO_FAIXA_KHZ = 10

# Quantas temperaturas mostrar nos histogramas
N_TEMPS_HIST = 6
SEED_TEMPS = 42


# ============================================================
# 3) CONFIGURAÇÃO VISUAL DOS GRÁFICOS
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 24,
    "axes.labelsize": 26,
    "axes.titlesize": 26,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "legend.fontsize": 18,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# 4) FUNÇÕES BÁSICAS
# ============================================================

def extract_freq_hz(col):
    """Extrai a frequência de colunas no formato f_30000Hz."""
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    """Seleciona colunas de frequência dentro da faixa escolhida."""
    cols, freqs = [], []

    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            f_khz = f / 1e3
            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    if len(cols) == 0:
        return [], np.array([])

    order = np.argsort(freqs)
    cols = [cols[i] for i in order]
    freqs = np.array(freqs)[order]

    return cols, freqs


def moving_average(arr, win):
    """Média móvel simples para suavizar a curva."""
    arr = np.asarray(arr, dtype=float)

    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")

    if len(smooth) > len(arr):
        smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr):
        smooth = np.pad(smooth, (0, len(arr) - len(smooth)), mode="edge")

    return smooth


def add_extra_features_matrix(X):
    """
    Mantém a lógica do seu RF ponto a ponto:
    usa a curva inteira + média + desvio padrão + amplitude.
    """
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    return np.hstack([X, mu, sd, amp])


def shift_interp(x, f, tau):
    """Deslocamento horizontal por interpolação."""
    f_shift = f + tau
    return np.interp(f, f_shift, x, left=x[0], right=x[-1])


def rmsd(y, ref):
    """RMSD: quanto menor, mais perto da referência."""
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    """
    CCDM = 1 - correlação de Pearson.
    Quanto menor, mais parecida é a forma da curva.
    """
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2))) + 1e-18

    corr = num / den
    return float(1 - corr)


def curva_referencia_saudavel(df, fcols):
    """
    Curva de referência = mediana das curvas sem dano na temperatura REF_TEMP.
    Se não existir REF_TEMP, usa a mediana de todas as curvas sem dano.
    """
    df_sem = df[df["falha"] == 0]
    X_sem = df_sem[fcols].to_numpy(float)

    pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)

    if len(pool) > 0:
        return np.median(pool, axis=0)

    print(f"⚠️ Não achei dados sem dano em {REF_TEMP}°C. Usando mediana geral sem dano.")
    return np.median(X_sem, axis=0)


# ============================================================
# 5) MÉTODO 1 — RF DIRETO PONTO A PONTO
# ============================================================

def compensar_rf_direto(df, fcols):
    """
    Treina o RF apenas com dados sem dano.
    Entrada: curva medida + features simples + temperatura.
    Saída aprendida: correção necessária para levar a curva sem dano até a referência.
    """
    df_sem = df[df["falha"] == 0]

    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    y_ref = curva_referencia_saudavel(df, fcols)

    # O alvo é o resíduo térmico aprendido nos dados saudáveis
    Y_target = y_ref[None, :] - X_sem

    X_aug = add_extra_features_matrix(X_sem)
    X_in = np.hstack([X_aug, T_sem.reshape(-1, 1)])

    rf = RandomForestRegressor(**RF_COMP_POINT_PARAMS)
    rf.fit(X_in, Y_target)

    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)

    X_aug_all = add_extra_features_matrix(X_all)
    X_in_all = np.hstack([X_aug_all, T_all.reshape(-1, 1)])

    Y_comp = X_all + rf.predict(X_in_all)

    for i in range(len(Y_comp)):
        Y_comp[i] = moving_average(Y_comp[i], SMOOTH_WIN)

    df2 = df.copy()
    df2[fcols] = Y_comp

    return df2, y_ref


# ============================================================
# 6) MÉTODO 2 — PARK
# ============================================================

def park_single(x, ref, fHz):
    """
    Park simplificado:
    testa vários deslocamentos horizontais e escolhe aquele que minimiza o erro
    após aplicar também um deslocamento vertical médio.
    """
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    for tau in np.linspace(-tau_max, tau_max, PARK_NSTEPS):
        x_shift = shift_interp(x, fHz, tau)
        dS = np.mean(ref - x_shift)
        err = np.sum((ref - (x_shift + dS)) ** 2)

        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y = shift_interp(x, fHz, best_tau) + best_dS
    y = moving_average(y, PARK_SMOOTH_WIN)

    return y


def compensar_park(df, fcols, fHz):
    y_ref = curva_referencia_saudavel(df, fcols)

    X_all = df[fcols].to_numpy(float)
    Y = np.zeros_like(X_all)

    for i in range(len(X_all)):
        Y[i] = park_single(X_all[i], y_ref, fHz)

    df2 = df.copy()
    df2[fcols] = Y

    return df2, y_ref


# ============================================================
# 7) MÉTRICAS
# ============================================================

def calcular_metricas(df_comp, fcols, y_ref):
    X = df_comp[fcols].to_numpy(float)

    df2 = df_comp[["temperatura_c", "falha"]].copy()
    df2["RMSD"] = [rmsd(x, y_ref) for x in X]
    df2["CCDM"] = [ccdm(x, y_ref) for x in X]

    return df2


def resumir_metricas_por_dano(df_metricas, metodo, fmin_khz, fmax_khz):
    """Resumo médio por dano para uma faixa e um método."""
    rows = []

    for d in sorted(df_metricas["falha"].unique()):
        sub = df_metricas[df_metricas["falha"] == d]

        rows.append({
            "faixa_min_khz": fmin_khz,
            "faixa_max_khz": fmax_khz,
            "largura_khz": fmax_khz - fmin_khz,
            "metodo": metodo,
            "falha": int(d),
            "RMSD_medio": sub["RMSD"].mean(),
            "RMSD_std": sub["RMSD"].std(),
            "CCDM_medio": sub["CCDM"].mean(),
            "CCDM_std": sub["CCDM"].std(),
            "n_amostras": len(sub),
        })

    return rows


# ============================================================
# 8) HISTOGRAMA GRANDE — RF vs PARK
# ============================================================

def histograma_rf_park_temperaturas_validas_2painel(
    df_rf,
    df_park,
    fmin_khz,
    fmax_khz,
    n_temps=6,
    seed=42,
    output_dir=PASTA_SAIDA,
    nome_extra=""
):
    """
    Histograma grande com RMSD e CCDM.
    Mostra RF e Park para dano 0, 1 e 2 em algumas temperaturas válidas.
    """

    required_cols = ["falha", "temperatura_c", "RMSD", "CCDM"]
    for name, data in [("df_rf", df_rf), ("df_park", df_park)]:
        missing = [c for c in required_cols if c not in data.columns]
        if missing:
            raise ValueError(f"{name} está sem as colunas: {missing}")

    danos = [0, 1, 2]

    temps_validas = None
    for data in [df_rf, df_park]:
        sets = []
        for d in danos:
            sets.append(set(data.loc[data["falha"] == d, "temperatura_c"].unique()))
        validas_data = set.intersection(*sets)
        temps_validas = validas_data if temps_validas is None else temps_validas & validas_data

    temps_validas = sorted(list(temps_validas))

    if len(temps_validas) == 0:
        raise ValueError("Nenhuma temperatura possui os três danos nos dois métodos.")

    if len(temps_validas) > n_temps:
        rng = np.random.default_rng(seed)
        temps_validas = sorted(rng.choice(temps_validas, n_temps, replace=False))

    colors = ["tab:blue", "tab:orange", "tab:red"]

    x = np.arange(len(temps_validas))

    bar_w = 0.12
    gap = 0.10
    rf_offsets = np.array([0, 1, 2]) * bar_w
    pk_offsets = (3 * bar_w + gap) + np.array([0, 1, 2]) * bar_w
    x_center = x + ((rf_offsets.mean() + pk_offsets.mean()) / 2)

    def medias(df, metric):
        out = {d: [] for d in danos}
        for d in danos:
            for T in temps_validas:
                mask = (df["falha"] == d) & np.isclose(df["temperatura_c"], T)
                out[d].append(df.loc[mask, metric].mean() if np.any(mask) else np.nan)
        return out

    rmsd_rf = medias(df_rf, "RMSD")
    rmsd_pk = medias(df_park, "RMSD")
    ccdm_rf = medias(df_rf, "CCDM")
    ccdm_pk = medias(df_park, "CCDM")

    fig, axes = plt.subplots(1, 2, figsize=(24, 8.5), dpi=300)

    for ax in axes:
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", labelsize=22)

    # --------------------------
    # RMSD
    # --------------------------
    ax = axes[0]

    for i, d in enumerate(danos):
        ax.bar(
            x + rf_offsets[i], rmsd_rf[d],
            width=bar_w,
            color=colors[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.9,
            label=f"RF ponto a ponto — dano {d}"
        )

        ax.bar(
            x + pk_offsets[i], rmsd_pk[d],
            width=bar_w,
            color=colors[i],
            alpha=1.0,
            edgecolor="black",
            linewidth=0.9,
            label=f"Park — dano {d}"
        )

    ax.set_ylabel("RMSD", fontsize=28)
    ax.set_xlabel("Temperatura (°C)", fontsize=28, labelpad=12)
    ax.set_xticks(x_center)
    ax.set_xticklabels([f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in temps_validas])
    ax.set_title("(a) RMSD", fontsize=28, pad=16)

    # --------------------------
    # CCDM
    # --------------------------
    ax = axes[1]

    for i, d in enumerate(danos):
        ax.bar(
            x + rf_offsets[i], ccdm_rf[d],
            width=bar_w,
            color=colors[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.9,
            label=f"RF ponto a ponto — dano {d}"
        )

        ax.bar(
            x + pk_offsets[i], ccdm_pk[d],
            width=bar_w,
            color=colors[i],
            alpha=1.0,
            edgecolor="black",
            linewidth=0.9,
            label=f"Park — dano {d}"
        )

    ax.set_ylabel("CCDM", fontsize=28)
    ax.set_xlabel("Temperatura (°C)", fontsize=28, labelpad=12)
    ax.set_xticks(x_center)
    ax.set_xticklabels([f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in temps_validas])
    ax.set_title("(b) CCDM", fontsize=28, pad=16)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=3,
        frameon=True,
        fontsize=18,
        bbox_to_anchor=(0.5, -0.08)
    )

    fig.suptitle(
        f"RF ponto a ponto vs Park — {fmin_khz:.0f}–{fmax_khz:.0f} kHz — referência {REF_TEMP}°C",
        fontsize=30,
        y=1.02
    )

    fig.tight_layout(rect=[0, 0.08, 1, 0.95])

    nome_base = f"hist_rf_park_{fmin_khz:.0f}_{fmax_khz:.0f}kHz{nome_extra}"
    png_path = os.path.join(output_dir, nome_base + ".png")
    pdf_path = os.path.join(output_dir, nome_base + ".pdf")

    fig.savefig(png_path, dpi=600, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    plt.show()

    print(f"✅ Histograma salvo em:\n{png_path}\n{pdf_path}")

    return fig


# ============================================================
# 9) EXECUTAR UMA FAIXA ESPECÍFICA
# ============================================================

def executar_uma_faixa(df_base, fmin_khz, fmax_khz, gerar_histograma=False):
    """Roda RF e Park para uma faixa específica."""

    fcols, fHz = get_freq_columns(df_base, fmin_khz, fmax_khz)

    if len(fcols) < 5:
        raise ValueError(f"Poucas colunas na faixa {fmin_khz}-{fmax_khz} kHz.")

    df_use = df_base[["temperatura_c", "falha"] + fcols].copy()

    print(f"\n🔹 Rodando faixa {fmin_khz:.0f}-{fmax_khz:.0f} kHz | {len(fcols)} pontos de frequência")

    t0 = time.time()

    df_rf_comp, yref_rf = compensar_rf_direto(df_use, fcols)
    df_park_comp, yref_park = compensar_park(df_use, fcols, fHz)

    df_rf_met = calcular_metricas(df_rf_comp, fcols, yref_rf)
    df_park_met = calcular_metricas(df_park_comp, fcols, yref_park)

    dt = time.time() - t0
    print(f"✅ Concluído em {dt:.1f} s")

    resumo = []
    resumo += resumir_metricas_por_dano(df_rf_met, "RF_ponto_a_ponto", fmin_khz, fmax_khz)
    resumo += resumir_metricas_por_dano(df_park_met, "Park", fmin_khz, fmax_khz)

    df_resumo = pd.DataFrame(resumo)

    if gerar_histograma:
        histograma_rf_park_temperaturas_validas_2painel(
            df_rf_met,
            df_park_met,
            fmin_khz=fmin_khz,
            fmax_khz=fmax_khz,
            n_temps=N_TEMPS_HIST,
            seed=SEED_TEMPS,
            output_dir=PASTA_SAIDA
        )

    return df_rf_met, df_park_met, df_resumo


# ============================================================
# 10) CRITÉRIO PARA ESCOLHER MELHOR FAIXA
# ============================================================

def _zscore_col(s):
    s = pd.Series(s, dtype=float)
    std = s.std()
    if std == 0 or np.isnan(std):
        return s * 0.0
    return (s - s.mean()) / std


def montar_tabela_ranking(df_resumo):
    """
    Monta ranking por método e faixa.

    Ideia do score:
    - Queremos RMSD e CCDM baixos no dano 0.
    - Queremos separação grande entre D0, D1 e D2.
    - Penalizamos inversão, quando D1 não fica maior que D0 ou D2 não fica maior que D1.

    Score menor = melhor faixa.
    """

    rows = []

    for (fmin, fmax, metodo), g in df_resumo.groupby(["faixa_min_khz", "faixa_max_khz", "metodo"]):
        gd = g.set_index("falha")

        if not all(d in gd.index for d in [0, 1, 2]):
            continue

        rmsd0 = gd.loc[0, "RMSD_medio"]
        rmsd1 = gd.loc[1, "RMSD_medio"]
        rmsd2 = gd.loc[2, "RMSD_medio"]

        ccdm0 = gd.loc[0, "CCDM_medio"]
        ccdm1 = gd.loc[1, "CCDM_medio"]
        ccdm2 = gd.loc[2, "CCDM_medio"]

        sep_rmsd_10 = rmsd1 - rmsd0
        sep_rmsd_21 = rmsd2 - rmsd1
        sep_ccdm_10 = ccdm1 - ccdm0
        sep_ccdm_21 = ccdm2 - ccdm1

        inversao_rmsd = int(sep_rmsd_10 <= 0) + int(sep_rmsd_21 <= 0)
        inversao_ccdm = int(sep_ccdm_10 <= 0) + int(sep_ccdm_21 <= 0)

        rows.append({
            "faixa_min_khz": fmin,
            "faixa_max_khz": fmax,
            "metodo": metodo,
            "RMSD_D0": rmsd0,
            "RMSD_D1": rmsd1,
            "RMSD_D2": rmsd2,
            "CCDM_D0": ccdm0,
            "CCDM_D1": ccdm1,
            "CCDM_D2": ccdm2,
            "sep_RMSD_D1_D0": sep_rmsd_10,
            "sep_RMSD_D2_D1": sep_rmsd_21,
            "sep_CCDM_D1_D0": sep_ccdm_10,
            "sep_CCDM_D2_D1": sep_ccdm_21,
            "inversoes_RMSD": inversao_rmsd,
            "inversoes_CCDM": inversao_ccdm,
            "inversoes_total": inversao_rmsd + inversao_ccdm,
        })

    rank = pd.DataFrame(rows)

    if len(rank) == 0:
        raise ValueError("Não foi possível montar ranking. Verifique se existem danos 0, 1 e 2.")

    # Quanto menor RMSD_D0 e CCDM_D0, melhor.
    # Quanto maior separação entre danos, melhor.
    # Como o score será minimizado, separações entram com sinal negativo.
    rank["score"] = (
        1.00 * _zscore_col(rank["RMSD_D0"]) +
        1.00 * _zscore_col(rank["CCDM_D0"]) -
        0.50 * _zscore_col(rank["sep_RMSD_D1_D0"]) -
        0.50 * _zscore_col(rank["sep_RMSD_D2_D1"]) -
        0.50 * _zscore_col(rank["sep_CCDM_D1_D0"]) -
        0.50 * _zscore_col(rank["sep_CCDM_D2_D1"]) +
        1.50 * rank["inversoes_total"]
    )

    rank = rank.sort_values("score", ascending=True).reset_index(drop=True)

    return rank


# ============================================================
# 11) VARREDURA EM TODA A FAIXA
# ============================================================

def gerar_faixas(fmin_global, fmax_global, largura, passo):
    """Gera faixas do tipo 30-40, 35-45, 40-50..."""
    faixas = []
    ini = fmin_global

    while ini + largura <= fmax_global:
        faixas.append((float(ini), float(ini + largura)))
        ini += passo

    return faixas


def varrer_faixas():
    print("🔹 Carregando base...")
    df_base = pd.read_pickle(ARQ_BASE)

    required = ["temperatura_c", "falha"]
    missing = [c for c in required if c not in df_base.columns]
    if missing:
        raise ValueError(f"A base não tem as colunas obrigatórias: {missing}")

    faixas = gerar_faixas(
        FREQ_MIN_GLOBAL_KHZ,
        FREQ_MAX_GLOBAL_KHZ,
        LARGURA_FAIXA_KHZ,
        PASSO_FAIXA_KHZ
    )

    print(f"🔎 Total de faixas a testar: {len(faixas)}")

    todos_resumos = []
    erros = []

    for fmin, fmax in faixas:
        try:
            _, _, df_resumo = executar_uma_faixa(
                df_base,
                fmin,
                fmax,
                gerar_histograma=False
            )
            todos_resumos.append(df_resumo)

        except Exception as e:
            print(f"⚠️ Erro na faixa {fmin}-{fmax} kHz: {e}")
            erros.append({"faixa_min_khz": fmin, "faixa_max_khz": fmax, "erro": str(e)})

    if len(todos_resumos) == 0:
        raise RuntimeError("Nenhuma faixa foi processada com sucesso.")

    df_resumo_total = pd.concat(todos_resumos, ignore_index=True)
    df_ranking = montar_tabela_ranking(df_resumo_total)

    path_resumo = os.path.join(PASTA_SAIDA, "resumo_metricas_todas_faixas.csv")
    path_ranking = os.path.join(PASTA_SAIDA, "ranking_melhores_faixas.csv")
    path_erros = os.path.join(PASTA_SAIDA, "erros_varredura.csv")

    df_resumo_total.to_csv(path_resumo, index=False)
    df_ranking.to_csv(path_ranking, index=False)

    if len(erros) > 0:
        pd.DataFrame(erros).to_csv(path_erros, index=False)

    print("\n✅ Varredura finalizada.")
    print(f"Resumo salvo em: {path_resumo}")
    print(f"Ranking salvo em: {path_ranking}")

    print("\n🏆 TOP 10 melhores faixas pelo critério combinado:")
    print(df_ranking.head(10).to_string(index=False))

    return df_base, df_resumo_total, df_ranking


# ============================================================
# 12) PLOT DO RANKING DAS FAIXAS
# ============================================================

def plot_ranking_faixas(df_ranking, top_n=15):
    """Plota as melhores faixas pelo score combinado."""

    top = df_ranking.head(top_n).copy()
    top["faixa"] = (
        top["faixa_min_khz"].astype(int).astype(str)
        + "–" +
        top["faixa_max_khz"].astype(int).astype(str)
        + " kHz\n" +
        top["metodo"]
    )

    fig, ax = plt.subplots(figsize=(18, 8), dpi=300)

    ax.bar(top["faixa"], top["score"], edgecolor="black", linewidth=0.9)
    ax.set_ylabel("Score combinado menor = melhor", fontsize=26)
    ax.set_xlabel("Faixa e método", fontsize=26)
    ax.set_title("Ranking das melhores faixas de frequência", fontsize=28, pad=16)
    ax.tick_params(axis="x", rotation=45, labelsize=18)
    ax.tick_params(axis="y", labelsize=22)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()

    png_path = os.path.join(PASTA_SAIDA, "ranking_melhores_faixas.png")
    pdf_path = os.path.join(PASTA_SAIDA, "ranking_melhores_faixas.pdf")

    fig.savefig(png_path, dpi=600, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    plt.show()

    print(f"✅ Ranking salvo em:\n{png_path}\n{pdf_path}")

    return fig

# ============================================================
# CÓDIGO ADICIONAL — PLOTAR RMSD E CCDM DE TODAS AS FAIXAS
# ============================================================
# Use este código DEPOIS de rodar o código principal da varredura.
# Ele lê o arquivo:
#   resultados_varredura_rf_park/resumo_metricas_todas_faixas.csv
#
# E gera gráficos grandes para usar em slides/artigo:
#   1) RMSD médio por faixa — RF e Park separados
#   2) CCDM médio por faixa — RF e Park separados
#   3) RMSD médio comparando RF vs Park para cada dano
#   4) CCDM médio comparando RF vs Park para cada dano
#   5) Separação entre danos por faixa
#   6) Heatmaps de RMSD e CCDM por método/dano/faixa
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1) CONFIGURAÇÕES
# ------------------------------------------------------------

PASTA_SAIDA = "resultados_varredura_rf_park"
ARQ_RESUMO = os.path.join(PASTA_SAIDA, "resumo_metricas_todas_faixas.csv")
PASTA_GRAFICOS = os.path.join(PASTA_SAIDA, "graficos_todas_as_faixas")
os.makedirs(PASTA_GRAFICOS, exist_ok=True)

# Escolha True para salvar também em PDF
SALVAR_PDF = True

# Visual maior, bom para apresentação/artigo
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 22,
    "axes.labelsize": 26,
    "axes.titlesize": 28,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Nomes bonitos para legenda
NOME_METODO = {
    "RF_ponto_a_ponto": "RF ponto a ponto",
    "Park": "Park"
}

# Cores por dano
CORES_DANO = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:red"
}

# Estilos por método
ESTILO_METODO = {
    "RF_ponto_a_ponto": "-",
    "Park": "--"
}

MARCADORES_METODO = {
    "RF_ponto_a_ponto": "o",
    "Park": "s"
}


# ------------------------------------------------------------
# 2) FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def preparar_resumo(path=ARQ_RESUMO):
    """Carrega o CSV da varredura e cria colunas úteis para plot."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Não encontrei o arquivo {path}.\n"
            "Rode primeiro o código principal da varredura para gerar o CSV."
        )

    df = pd.read_csv(path)

    cols_obrigatorias = [
        "faixa_min_khz", "faixa_max_khz", "metodo", "falha",
        "RMSD_medio", "RMSD_std", "CCDM_medio", "CCDM_std"
    ]
    faltando = [c for c in cols_obrigatorias if c not in df.columns]
    if faltando:
        raise ValueError(f"O CSV não tem as colunas obrigatórias: {faltando}")

    df["faixa_centro_khz"] = 0.5 * (df["faixa_min_khz"] + df["faixa_max_khz"])
    df["faixa_label"] = (
        df["faixa_min_khz"].astype(int).astype(str)
        + "–"
        + df["faixa_max_khz"].astype(int).astype(str)
        + " kHz"
    )

    df = df.sort_values(["faixa_min_khz", "faixa_max_khz", "metodo", "falha"]).reset_index(drop=True)
    return df


def salvar_fig(fig, nome_base):
    """Salva figura em PNG e, opcionalmente, PDF."""
    png_path = os.path.join(PASTA_GRAFICOS, nome_base + ".png")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    if SALVAR_PDF:
        pdf_path = os.path.join(PASTA_GRAFICOS, nome_base + ".pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"✅ Salvo:\n{png_path}\n{pdf_path}")
    else:
        print(f"✅ Salvo:\n{png_path}")


def estilo_eixos(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=20)


def erro_valido(y, yerr):
    """Evita erro NaN no errorbar."""
    y = np.asarray(y, dtype=float)
    yerr = np.asarray(yerr, dtype=float)
    yerr = np.where(np.isfinite(yerr), yerr, 0.0)
    return y, yerr


# ------------------------------------------------------------
# 3) PLOT 1 — RMSD OU CCDM POR FAIXA, SEPARADO POR MÉTODO
# ------------------------------------------------------------

def plot_metrica_por_faixa_por_metodo(df, metrica="RMSD"):
    """
    Gera 1 figura com 2 painéis:
    - esquerda: RF ponto a ponto
    - direita: Park

    Cada linha representa um dano.
    """
    col_media = f"{metrica}_medio"
    col_std = f"{metrica}_std"

    metodos = ["RF_ponto_a_ponto", "Park"]
    danos = sorted(df["falha"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(24, 8), dpi=300, sharey=True)

    for ax, metodo in zip(axes, metodos):
        sub_met = df[df["metodo"] == metodo]

        for dano in danos:
            sub = sub_met[sub_met["falha"] == dano].sort_values("faixa_centro_khz")

            y, yerr = erro_valido(sub[col_media], sub[col_std])

            ax.errorbar(
                sub["faixa_centro_khz"],
                y,
                yerr=yerr,
                marker="o",
                linewidth=2.8,
                markersize=8,
                capsize=4,
                color=CORES_DANO.get(int(dano), None),
                label=f"Dano {int(dano)}"
            )

        ax.set_title(NOME_METODO.get(metodo, metodo), fontsize=28, pad=14)
        ax.set_xlabel("Centro da faixa de frequência (kHz)", fontsize=26)
        estilo_eixos(ax)

    axes[0].set_ylabel(metrica, fontsize=28)

    fig.suptitle(f"{metrica} médio em todas as faixas", fontsize=32, y=1.03)
    axes[1].legend(frameon=True, fontsize=18, loc="best")

    fig.tight_layout()
    salvar_fig(fig, f"{metrica.lower()}_medio_todas_faixas_por_metodo")
    plt.show()

    return fig


# ------------------------------------------------------------
# 4) PLOT 2 — COMPARAÇÃO RF vs PARK PARA CADA DANO
# ------------------------------------------------------------

def plot_metrica_rf_vs_park_por_dano(df, metrica="RMSD"):
    """
    Gera 1 figura com 3 painéis:
    - dano 0
    - dano 1
    - dano 2

    Em cada painel aparecem RF ponto a ponto e Park.
    """
    col_media = f"{metrica}_medio"
    col_std = f"{metrica}_std"

    danos = sorted(df["falha"].unique())
    metodos = ["RF_ponto_a_ponto", "Park"]

    fig, axes = plt.subplots(1, len(danos), figsize=(8 * len(danos), 7.5), dpi=300, sharey=True)

    if len(danos) == 1:
        axes = [axes]

    for ax, dano in zip(axes, danos):
        sub_dano = df[df["falha"] == dano]

        for metodo in metodos:
            sub = sub_dano[sub_dano["metodo"] == metodo].sort_values("faixa_centro_khz")
            y, yerr = erro_valido(sub[col_media], sub[col_std])

            ax.errorbar(
                sub["faixa_centro_khz"],
                y,
                yerr=yerr,
                linestyle=ESTILO_METODO.get(metodo, "-"),
                marker=MARCADORES_METODO.get(metodo, "o"),
                linewidth=2.8,
                markersize=8,
                capsize=4,
                label=NOME_METODO.get(metodo, metodo)
            )

        ax.set_title(f"Dano {int(dano)}", fontsize=28, pad=14)
        ax.set_xlabel("Centro da faixa (kHz)", fontsize=26)
        estilo_eixos(ax)

    axes[0].set_ylabel(metrica, fontsize=28)
    axes[-1].legend(frameon=True, fontsize=18, loc="best")

    fig.suptitle(f"Comparação RF vs Park — {metrica} em todas as faixas", fontsize=32, y=1.03)
    fig.tight_layout()

    salvar_fig(fig, f"{metrica.lower()}_rf_vs_park_por_dano_todas_faixas")
    plt.show()

    return fig


# ------------------------------------------------------------
# 5) PLOT 3 — SEPARAÇÃO ENTRE DANOS POR FAIXA
# ------------------------------------------------------------

def calcular_separacoes(df):
    """
    Calcula separações médias entre danos:
    - D1 - D0
    - D2 - D1
    - D2 - D0

    Isso ajuda a ver se a faixa preserva assinatura de dano.
    """
    rows = []

    for (fmin, fmax, metodo), g in df.groupby(["faixa_min_khz", "faixa_max_khz", "metodo"]):
        gd = g.set_index("falha")

        if not all(d in gd.index for d in [0, 1, 2]):
            continue

        for metrica in ["RMSD", "CCDM"]:
            v0 = gd.loc[0, f"{metrica}_medio"]
            v1 = gd.loc[1, f"{metrica}_medio"]
            v2 = gd.loc[2, f"{metrica}_medio"]

            rows.append({
                "faixa_min_khz": fmin,
                "faixa_max_khz": fmax,
                "faixa_centro_khz": 0.5 * (fmin + fmax),
                "metodo": metodo,
                "metrica": metrica,
                "sep_D1_D0": v1 - v0,
                "sep_D2_D1": v2 - v1,
                "sep_D2_D0": v2 - v0,
                "inversao_D1_D0": int((v1 - v0) <= 0),
                "inversao_D2_D1": int((v2 - v1) <= 0),
            })

    return pd.DataFrame(rows)


def plot_separacao_danos(df, metrica="RMSD"):
    """
    Plota separação entre danos em todas as faixas.

    Interpretação:
    - Valores positivos são bons.
    - Se D1-D0 ou D2-D1 ficar negativo, há inversão da ordem esperada.
    """
    sep = calcular_separacoes(df)
    sep = sep[sep["metrica"] == metrica].copy()

    metodos = ["RF_ponto_a_ponto", "Park"]
    variaveis = ["sep_D1_D0", "sep_D2_D1", "sep_D2_D0"]
    labels = {
        "sep_D1_D0": "Dano 1 − Dano 0",
        "sep_D2_D1": "Dano 2 − Dano 1",
        "sep_D2_D0": "Dano 2 − Dano 0"
    }

    fig, axes = plt.subplots(1, 2, figsize=(24, 8), dpi=300, sharey=True)

    for ax, metodo in zip(axes, metodos):
        sub_m = sep[sep["metodo"] == metodo].sort_values("faixa_centro_khz")

        for var in variaveis:
            ax.plot(
                sub_m["faixa_centro_khz"],
                sub_m[var],
                marker="o",
                linewidth=2.8,
                markersize=8,
                label=labels[var]
            )

        ax.axhline(0, color="black", linewidth=1.2)
        ax.set_title(NOME_METODO.get(metodo, metodo), fontsize=28, pad=14)
        ax.set_xlabel("Centro da faixa (kHz)", fontsize=26)
        estilo_eixos(ax)

    axes[0].set_ylabel(f"Separação em {metrica}", fontsize=28)
    axes[1].legend(frameon=True, fontsize=18, loc="best")

    fig.suptitle(f"Separação entre danos por faixa — {metrica}", fontsize=32, y=1.03)
    fig.tight_layout()

    salvar_fig(fig, f"separacao_danos_{metrica.lower()}_todas_faixas")
    plt.show()

    return fig


# ------------------------------------------------------------
# 6) PLOT 4 — HEATMAP DE MÉTRICA POR FAIXA E DANO
# ------------------------------------------------------------

def plot_heatmap_metrica(df, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    """
    Heatmap simples:
    - eixo x: faixa de frequência
    - eixo y: dano
    - cor: RMSD ou CCDM médio
    """
    col_media = f"{metrica}_medio"

    sub = df[df["metodo"] == metodo].copy()
    sub = sub.sort_values(["faixa_min_khz", "falha"])

    pivot = sub.pivot_table(
        index="falha",
        columns="faixa_label",
        values=col_media,
        aggfunc="mean"
    )

    # Reordena colunas pela frequência mínima
    ordem = (
        sub[["faixa_label", "faixa_min_khz"]]
        .drop_duplicates()
        .sort_values("faixa_min_khz")["faixa_label"]
        .tolist()
    )
    pivot = pivot[ordem]

    fig, ax = plt.subplots(figsize=(max(14, 0.9 * len(ordem)), 6.5), dpi=300)

    im = ax.imshow(pivot.values, aspect="auto")

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([f"Dano {int(d)}" for d in pivot.index])

    ax.set_xlabel("Faixa de frequência", fontsize=26)
    ax.set_ylabel("Estado estrutural", fontsize=26)
    ax.set_title(f"{metrica} médio — {NOME_METODO.get(metodo, metodo)}", fontsize=28, pad=16)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metrica, fontsize=24)
    cbar.ax.tick_params(labelsize=18)

    # Escreve valores dentro das células
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:.3g}", ha="center", va="center", fontsize=13)

    fig.tight_layout()

    nome_metodo = metodo.lower().replace("_", "-")
    salvar_fig(fig, f"heatmap_{metrica.lower()}_{nome_metodo}_todas_faixas")
    plt.show()

    return fig


# ------------------------------------------------------------
# 7) RODAR TODOS OS GRÁFICOS
# ------------------------------------------------------------

def plotar_tudo_todas_as_faixas():
    df = preparar_resumo(ARQ_RESUMO)

    # 1) RMSD e CCDM por faixa, com RF e Park separados
    plot_metrica_por_faixa_por_metodo(df, metrica="RMSD")
    plot_metrica_por_faixa_por_metodo(df, metrica="CCDM")

    # 2) RMSD e CCDM comparando RF vs Park dentro de cada dano
    plot_metrica_rf_vs_park_por_dano(df, metrica="RMSD")
    plot_metrica_rf_vs_park_por_dano(df, metrica="CCDM")

    # 3) Separação entre danos
    plot_separacao_danos(df, metrica="RMSD")
    plot_separacao_danos(df, metrica="CCDM")

    # 4) Heatmaps
    for metodo in ["RF_ponto_a_ponto", "Park"]:
        plot_heatmap_metrica(df, metrica="RMSD", metodo=metodo)
        plot_heatmap_metrica(df, metrica="CCDM", metodo=metodo)

    print("\n✅ Todos os gráficos de todas as faixas foram gerados.")
    print(f"📁 Pasta de saída: {PASTA_GRAFICOS}")

    return df

# ============================================================
# HEATMAP 3D — FREQUÊNCIA × TEMPERATURA × MÉTRICA
# COLE ESTE BLOCO NO SEU CÓDIGO
# ============================================================

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib import cm


def salvar_metricas_detalhadas_faixa(df_metricas, metodo, fmin_khz, fmax_khz):
    """
    Mantém as métricas amostra a amostra, sem perder a temperatura.

    Isso é o que faltava para fazer o heatmap 3D:
        frequência/faixa × temperatura × RMSD ou CCDM

    Entrada:
        df_metricas vem de calcular_metricas()
        metodo = "RF_ponto_a_ponto" ou "Park"
        fmin_khz, fmax_khz = limites da faixa analisada
    """

    df_out = df_metricas.copy()

    df_out["metodo"] = metodo
    df_out["faixa_min_khz"] = fmin_khz
    df_out["faixa_max_khz"] = fmax_khz
    df_out["faixa_centro_khz"] = (fmin_khz + fmax_khz) / 2
    df_out["largura_khz"] = fmax_khz - fmin_khz

    return df_out


def plot_heatmap3d_temperatura(
    df_metricas_todas,
    metrica="RMSD",
    metodo="RF_ponto_a_ponto",
    falha=None,
    output_dir=PASTA_SAIDA,
    nome_extra=""
):
    """
    Faz um heatmap 3D:

        eixo X = centro da faixa de frequência [kHz]
        eixo Y = temperatura [°C]
        eixo Z = RMSD ou CCDM
        cor    = RMSD ou CCDM

    Parâmetros:
        metrica = "RMSD" ou "CCDM"
        metodo  = "RF_ponto_a_ponto" ou "Park"
        falha   = None para todos os danos juntos,
                  ou 0, 1, 2 para plotar um dano específico.
    """

    dfp = df_metricas_todas.copy()

    dfp = dfp[dfp["metodo"] == metodo].copy()

    if falha is not None:
        dfp = dfp[dfp["falha"] == falha].copy()

    if dfp.empty:
        print(f"⚠️ Sem dados para método={metodo}, falha={falha}, métrica={metrica}")
        return None

    # Agrupa porque pode haver mais de uma amostra para a mesma temperatura/faixa.
    tabela = (
        dfp
        .groupby(["temperatura_c", "faixa_centro_khz"], as_index=False)[metrica]
        .mean()
        .pivot(index="temperatura_c", columns="faixa_centro_khz", values=metrica)
        .sort_index()
        .sort_index(axis=1)
    )

    X_vals = tabela.columns.to_numpy(dtype=float)
    Y_vals = tabela.index.to_numpy(dtype=float)

    X, Y = np.meshgrid(X_vals, Y_vals)
    Z = tabela.to_numpy(dtype=float)

    fig = plt.figure(figsize=(15, 10), dpi=300)
    ax = fig.add_subplot(111, projection="3d")

    surf = ax.plot_surface(
        X,
        Y,
        Z,
        cmap=cm.viridis,
        linewidth=0,
        antialiased=True,
        alpha=0.95
    )

    # Projeção tipo heatmap no chão do gráfico
    try:
        zmin = np.nanmin(Z)
        zmax = np.nanmax(Z)
        offset = zmin - 0.08 * (zmax - zmin + 1e-12)

        ax.contourf(
            X,
            Y,
            Z,
            zdir="z",
            offset=offset,
            levels=25,
            cmap=cm.viridis,
            alpha=0.95
        )

        ax.set_zlim(offset, zmax)
    except Exception:
        pass

    nome_metodo = {
        "RF_ponto_a_ponto": "RF ponto a ponto",
        "Park": "Park",
    }.get(metodo, metodo)

    if falha is None:
        titulo_falha = "todos os danos"
        nome_falha = "todos_danos"
    else:
        titulo_falha = f"dano {falha}"
        nome_falha = f"dano_{falha}"

    ax.set_title(
        f"Heatmap 3D — {metrica} em função da frequência e temperatura\n"
        f"{nome_metodo} — {titulo_falha}",
        fontsize=24,
        pad=22
    )

    ax.set_xlabel("Centro da faixa de frequência [kHz]", fontsize=20, labelpad=16)
    ax.set_ylabel("Temperatura [°C]", fontsize=20, labelpad=16)
    ax.set_zlabel(metrica, fontsize=20, labelpad=16)

    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    ax.tick_params(axis="z", labelsize=14)

    ax.view_init(elev=30, azim=-135)

    cbar = fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.12)
    cbar.set_label(metrica, fontsize=18)
    cbar.ax.tick_params(labelsize=14)

    fig.tight_layout()

    nome_base = f"heatmap3d_{metrica.lower()}_{metodo}_{nome_falha}{nome_extra}"

    png_path = os.path.join(output_dir, nome_base + ".png")
    pdf_path = os.path.join(output_dir, nome_base + ".pdf")

    fig.savefig(png_path, dpi=600, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    plt.show()

    print(f"✅ Heatmap 3D salvo em:\n{png_path}\n{pdf_path}")

    return fig


def plotar_todos_heatmaps3d_temperatura(df_metricas_todas, output_dir=PASTA_SAIDA):
    """
    Plota automaticamente:
        RF e Park
        RMSD e CCDM
        todos os danos juntos
        dano 0, dano 1 e dano 2 separados
    """

    metodos = ["RF_ponto_a_ponto", "Park"]
    metricas = ["RMSD", "CCDM"]

    falhas_disponiveis = sorted(df_metricas_todas["falha"].dropna().unique())

    for metodo in metodos:
        for metrica in metricas:

            # Todos os danos juntos
            plot_heatmap3d_temperatura(
                df_metricas_todas,
                metrica=metrica,
                metodo=metodo,
                falha=None,
                output_dir=output_dir
            )

            # Cada dano separado
            for f in falhas_disponiveis:
                plot_heatmap3d_temperatura(
                    df_metricas_todas,
                    metrica=metrica,
                    metodo=metodo,
                    falha=int(f),
                    output_dir=output_dir
                )


# ============================================================
# ALTERAÇÃO 1 — TROQUE SUA FUNÇÃO executar_uma_faixa POR ESTA
# ============================================================

def executar_uma_faixa(df_base, fmin_khz, fmax_khz, gerar_histograma=False):
    """Roda RF e Park para uma faixa específica."""

    fcols, fHz = get_freq_columns(df_base, fmin_khz, fmax_khz)

    if len(fcols) < 5:
        raise ValueError(f"Poucas colunas na faixa {fmin_khz}-{fmax_khz} kHz.")

    df_use = df_base[["temperatura_c", "falha"] + fcols].copy()

    print(f"\n🔹 Rodando faixa {fmin_khz:.0f}-{fmax_khz:.0f} kHz | {len(fcols)} pontos de frequência")

    t0 = time.time()

    df_rf_comp, yref_rf = compensar_rf_direto(df_use, fcols)
    df_park_comp, yref_park = compensar_park(df_use, fcols, fHz)

    df_rf_met = calcular_metricas(df_rf_comp, fcols, yref_rf)
    df_park_met = calcular_metricas(df_park_comp, fcols, yref_park)

    dt = time.time() - t0
    print(f"✅ Concluído em {dt:.1f} s")

    resumo = []
    resumo += resumir_metricas_por_dano(df_rf_met, "RF_ponto_a_ponto", fmin_khz, fmax_khz)
    resumo += resumir_metricas_por_dano(df_park_met, "Park", fmin_khz, fmax_khz)

    df_resumo = pd.DataFrame(resumo)

    # NOVO:
    # aqui ficam as métricas detalhadas com temperatura e faixa.
    # É isso que permite fazer o heatmap 3D.
    df_rf_det = salvar_metricas_detalhadas_faixa(
        df_rf_met,
        metodo="RF_ponto_a_ponto",
        fmin_khz=fmin_khz,
        fmax_khz=fmax_khz
    )

    df_park_det = salvar_metricas_detalhadas_faixa(
        df_park_met,
        metodo="Park",
        fmin_khz=fmin_khz,
        fmax_khz=fmax_khz
    )

    df_detalhado = pd.concat([df_rf_det, df_park_det], ignore_index=True)

    if gerar_histograma:
        histograma_rf_park_temperaturas_validas_2painel(
            df_rf_met,
            df_park_met,
            fmin_khz=fmin_khz,
            fmax_khz=fmax_khz,
            n_temps=N_TEMPS_HIST,
            seed=SEED_TEMPS,
            output_dir=PASTA_SAIDA
        )

    return df_rf_met, df_park_met, df_resumo, df_detalhado


# ============================================================
# ALTERAÇÃO 2 — TROQUE SUA FUNÇÃO varrer_faixas POR ESTA
# ============================================================

def varrer_faixas():
    print("🔹 Carregando base...")
    df_base = pd.read_pickle(ARQ_BASE)

    required = ["temperatura_c", "falha"]
    missing = [c for c in required if c not in df_base.columns]
    if missing:
        raise ValueError(f"A base não tem as colunas obrigatórias: {missing}")

    faixas = gerar_faixas(
        FREQ_MIN_GLOBAL_KHZ,
        FREQ_MAX_GLOBAL_KHZ,
        LARGURA_FAIXA_KHZ,
        PASSO_FAIXA_KHZ
    )

    print(f"🔎 Total de faixas a testar: {len(faixas)}")

    todos_resumos = []
    todos_detalhados = []
    erros = []

    for fmin, fmax in faixas:
        try:
            _, _, df_resumo, df_detalhado = executar_uma_faixa(
                df_base,
                fmin,
                fmax,
                gerar_histograma=False
            )

            todos_resumos.append(df_resumo)
            todos_detalhados.append(df_detalhado)

        except Exception as e:
            print(f"⚠️ Erro na faixa {fmin}-{fmax} kHz: {e}")
            erros.append({"faixa_min_khz": fmin, "faixa_max_khz": fmax, "erro": str(e)})

    if len(todos_resumos) == 0:
        raise RuntimeError("Nenhuma faixa foi processada com sucesso.")

    df_resumo_total = pd.concat(todos_resumos, ignore_index=True)
    df_metricas_todas = pd.concat(todos_detalhados, ignore_index=True)

    df_ranking = montar_tabela_ranking(df_resumo_total)

    path_resumo = os.path.join(PASTA_SAIDA, "resumo_metricas_todas_faixas.csv")
    path_ranking = os.path.join(PASTA_SAIDA, "ranking_melhores_faixas.csv")
    path_detalhado = os.path.join(PASTA_SAIDA, "metricas_detalhadas_temperatura_faixa.csv")
    path_erros = os.path.join(PASTA_SAIDA, "erros_varredura.csv")

    df_resumo_total.to_csv(path_resumo, index=False)
    df_ranking.to_csv(path_ranking, index=False)
    df_metricas_todas.to_csv(path_detalhado, index=False)

    if len(erros) > 0:
        pd.DataFrame(erros).to_csv(path_erros, index=False)

    print("\n✅ Varredura finalizada.")
    print(f"Resumo salvo em: {path_resumo}")
    print(f"Ranking salvo em: {path_ranking}")
    print(f"Métricas detalhadas salvas em: {path_detalhado}")

    print("\n🏆 TOP 10 melhores faixas pelo critério combinado:")
    print(df_ranking.head(10).to_string(index=False))

    return df_base, df_resumo_total, df_ranking, df_metricas_todas


# ============================================================
# EXECUÇÃO ÚNICA — RODA TUDO EM UM CÓDIGO SÓ
# ============================================================

if __name__ == "__main__":

    print("\n" + "=" * 90)
    print("ANÁLISE COMPLETA — RF PONTO A PONTO vs PARK")
    print("VARREDURA + HISTOGRAMAS + GRÁFICOS 2D + HEATMAPS + HEATMAPS 3D")
    print("=" * 90)

    # 1) Varre todas as faixas configuradas
    #    Para mudar 10, 20 ou 30 kHz, altere no começo:
    #       LARGURA_FAIXA_KHZ = 10, 20 ou 30
    #       PASSO_FAIXA_KHZ = 10, 20 ou 30
    df_base, df_resumo_total, df_ranking, df_metricas_todas = varrer_faixas()

    # 2) Plota ranking das melhores faixas
    plot_ranking_faixas(df_ranking, top_n=15)

    # 3) Plota os gráficos 2D de todas as faixas
    #    RMSD, CCDM, separações e heatmaps 2D
    try:
        plotar_tudo_todas_as_faixas()
    except Exception as e:
        print(f"⚠️ Não consegui gerar os gráficos 2D extras: {e}")

    # 4) Plota os heatmaps 3D com temperatura
    try:
        plotar_todos_heatmaps3d_temperatura(
            df_metricas_todas,
            output_dir=PASTA_SAIDA
        )
    except Exception as e:
        print(f"⚠️ Não consegui gerar os heatmaps 3D: {e}")

    # 5) Pega a melhor faixa do ranking geral
    melhor = df_ranking.iloc[0]
    melhor_fmin = float(melhor["faixa_min_khz"])
    melhor_fmax = float(melhor["faixa_max_khz"])

    print("\n🏆 Melhor faixa encontrada:")
    print(melhor.to_string())

    # 6) Recalcula a melhor faixa e gera o histograma grande RF vs Park
    df_rf_best, df_park_best, df_resumo_best, df_detalhado_best = executar_uma_faixa(
        df_base,
        melhor_fmin,
        melhor_fmax,
        gerar_histograma=True
    )

    # 7) Salva as métricas da melhor faixa
    df_rf_best.to_csv(
        os.path.join(PASTA_SAIDA, f"metricas_RF_melhor_faixa_{melhor_fmin:.0f}_{melhor_fmax:.0f}kHz.csv"),
        index=False
    )

    df_park_best.to_csv(
        os.path.join(PASTA_SAIDA, f"metricas_Park_melhor_faixa_{melhor_fmin:.0f}_{melhor_fmax:.0f}kHz.csv"),
        index=False
    )

    df_detalhado_best.to_csv(
        os.path.join(PASTA_SAIDA, f"metricas_detalhadas_melhor_faixa_{melhor_fmin:.0f}_{melhor_fmax:.0f}kHz.csv"),
        index=False
    )

    print("\n✅ Tudo finalizado.")
    print(f"📁 Pasta principal de saída: {PASTA_SAIDA}")
    print("\nArquivos principais:")
    print(f" - {os.path.join(PASTA_SAIDA, 'resumo_metricas_todas_faixas.csv')}")
    print(f" - {os.path.join(PASTA_SAIDA, 'ranking_melhores_faixas.csv')}")
    print(f" - {os.path.join(PASTA_SAIDA, 'metricas_detalhadas_temperatura_faixa.csv')}")
